# bd.ipynb - 싸움 / 스토킹 행동 판정 전용 노트북

이 노트북은 기존 b파트와 섞지 않고, tracking 결과 JSON을 입력으로 받아 다음 두 행동을 판정합니다.

- **fight**: 사람 bbox에서 손/발 후보 영역을 근사하고, 해당 영역이 상대 bbox와 겹치거나 가까워지는 접촉 후보가 2회 이상 반복되면 싸움 후보로 저장
- **stalking**: 한 사람이 다른 사람의 뒤쪽 위치에서 일정 거리로 같은 방향 이동을 지속하면 스토킹 후보로 저장

출력은 `outputs/prediction/bd_behavior_prediction.json`에 저장됩니다.


In [10]:
from pathlib import Path
from itertools import combinations, permutations
from collections import defaultdict
import json
import math

# ==============================
# 경로 설정
# ==============================
# bd.ipynb를 프로젝트 루트 또는 코드 폴더에 두고 실행하는 기준입니다.
BASE_DIR = Path.cwd()

# 필요하면 여기만 본인 tracking json 이름으로 바꾸면 됩니다.
INPUT_JSON = BASE_DIR / "cf2"

OUTPUT_DIR = BASE_DIR / "outputs" / "prediction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON = OUTPUT_DIR / "bd_behavior_prediction.json"

print("BASE_DIR:", BASE_DIR)
print("INPUT_JSON:", INPUT_JSON)
print("OUTPUT_JSON:", OUTPUT_JSON)


BASE_DIR: C:\Users\우성한
INPUT_JSON: C:\Users\우성한\cf2
OUTPUT_JSON: C:\Users\우성한\outputs\prediction\bd_behavior_prediction.json


In [11]:
# ==============================
# 파라미터
# ==============================

MIN_TRACK_LENGTH = 8

# 싸움 판정
FIGHT_BODY_DISTANCE_TH = 220       # 두 사람 중심 거리 기준
FIGHT_PART_TOUCH_MARGIN = 25       # 손/발 후보 bbox와 상대 bbox 간 허용 거리
FIGHT_MOTION_TH = 7                # 손/발 공격 후보로 보려는 프레임 간 이동량
FIGHT_WINDOW_SEC = 3.0             # 이 시간 안에서 반복 접촉을 확인
FIGHT_MIN_HIT_COUNT = 2            # 사용자 조건: 2회 이상 반복
FIGHT_MIN_SEPARATION_SEC = 0.35    # 같은 접촉이 중복 카운트되지 않도록 최소 시간 간격

# 스토킹 판정
STALKING_MIN_DIST = 70
STALKING_MAX_DIST = 420
STALKING_DIR_SIM_TH = 0.60
STALKING_BEHIND_DOT_TH = 15        # target 진행방향 기준 follower가 뒤에 있는 정도
STALKING_LATERAL_MAX = 220         # target 진행방향의 수직방향으로 너무 멀면 제외
STALKING_WINDOW_SEC = 5.0
STALKING_MIN_RATIO = 0.55
STALKING_MIN_SIGNAL_COUNT = 8
STALKING_DISTANCE_STD_TH = 120

# time_sec가 없는 json을 위한 기본 fps
DEFAULT_FPS = 30.0


In [12]:
# ==============================
# 기본 유틸
# ==============================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def bbox_center(b):
    x1, y1, x2, y2 = map(float, b)
    return [(x1 + x2) / 2, (y1 + y2) / 2]


def bbox_wh(b):
    x1, y1, x2, y2 = map(float, b)
    return max(1.0, x2 - x1), max(1.0, y2 - y1)


def foot_point(b):
    x1, y1, x2, y2 = map(float, b)
    return [(x1 + x2) / 2, y2]


def dist(p, q):
    return math.hypot(p[0] - q[0], p[1] - q[1])


def vector(p, q):
    return [q[0] - p[0], q[1] - p[1]]


def norm(v):
    return math.hypot(v[0], v[1])


def cosine_similarity(v1, v2):
    n1, n2 = norm(v1), norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return (v1[0] * v2[0] + v1[1] * v2[1]) / (n1 * n2)


def mean(xs):
    return sum(xs) / len(xs) if xs else 0.0


def std(xs):
    if len(xs) < 2:
        return 0.0
    m = mean(xs)
    return math.sqrt(sum((x - m) ** 2 for x in xs) / len(xs))


def box_distance(a, b):
    """두 bbox가 겹치면 0, 떨어져 있으면 최소 간격."""
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)
    dx = max(bx1 - ax2, ax1 - bx2, 0)
    dy = max(by1 - ay2, ay1 - by2, 0)
    return math.hypot(dx, dy)


def clamp_box(b):
    x1, y1, x2, y2 = b
    return [round(float(x1), 2), round(float(y1), 2), round(float(x2), 2), round(float(y2), 2)]


In [13]:
# ==============================
# 입력 JSON 파서
# ==============================
# 지원 형태 1: tracking list
# [ {"frame_id":0, "track_id":1, "class_name":"person", "bbox":[...]}, ... ]
#
# 지원 형태 2: cf 계열 frame dict
# {"video_name":"...", "frames":[{"frame_id":0, "objects":[{"class":"person", "bbox":[...]}]}]}
# 단, 스토킹/싸움은 track_id가 있어야 안정적입니다.

def normalize_records(raw):
    records = []
    video_name = "unknown"

    if isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            video_name = item.get("video_name", video_name)
            records.append(item)
        return video_name, records

    if isinstance(raw, dict):
        video_name = raw.get("video_name", video_name)
        frames = raw.get("frames", [])
        for fr in frames:
            frame_id = fr.get("frame_id", fr.get("frame", 0))
            time_sec = fr.get("time_sec", frame_id / DEFAULT_FPS)
            objects = fr.get("objects", [])
            for idx, obj in enumerate(objects):
                cls = obj.get("class_name", obj.get("class", obj.get("label", "")))
                bbox = obj.get("bbox")
                if bbox is None:
                    continue
                # track_id가 없으면 frame별 임시 id라 관계 추적은 약합니다.
                track_id = obj.get("track_id", obj.get("id", f"no_track_{idx}"))
                records.append({
                    "video_name": video_name,
                    "frame_id": frame_id,
                    "time_sec": time_sec,
                    "track_id": track_id,
                    "class_name": cls,
                    "confidence": obj.get("confidence", obj.get("conf")),
                    "bbox": bbox,
                })
        return video_name, records

    raise ValueError("지원하지 않는 JSON 구조입니다.")


def build_person_tracks(raw):
    video_name, records = normalize_records(raw)
    tracks = defaultdict(list)

    for item in records:
        cls = str(item.get("class_name", item.get("class", ""))).lower()
        if cls not in ["person", "human"]:
            continue

        bbox = item.get("bbox")
        tid = item.get("track_id")
        if bbox is None or tid is None:
            continue

        frame_id = int(item.get("frame_id", 0))
        time_sec = float(item.get("time_sec", frame_id / DEFAULT_FPS))
        tid = str(tid)

        tracks[tid].append({
            "video_name": item.get("video_name", video_name),
            "frame_id": frame_id,
            "time_sec": time_sec,
            "track_id": tid,
            "bbox": [float(x) for x in bbox],
            "center": bbox_center(bbox),
            "foot_point": foot_point(bbox),
            "confidence": item.get("confidence"),
        })

    cleaned = {}
    for tid, pts in tracks.items():
        pts = sorted(pts, key=lambda x: x["frame_id"])
        if len(pts) >= MIN_TRACK_LENGTH:
            cleaned[tid] = pts

    return video_name, cleaned


def build_frame_map(tracks):
    frame_map = defaultdict(dict)
    for tid, pts in tracks.items():
        for p in pts:
            frame_map[p["frame_id"]][tid] = p
    return frame_map


def get_point_by_frame(track_points):
    return {p["frame_id"]: p for p in track_points}


def prev_point(track_frame_map, frame_id):
    prev_frames = [fid for fid in track_frame_map if fid < frame_id]
    if not prev_frames:
        return None
    return track_frame_map[max(prev_frames)]


def compute_speed(track_frame_map, frame_id):
    cur = track_frame_map.get(frame_id)
    prev = prev_point(track_frame_map, frame_id)
    if cur is None or prev is None:
        return 0.0
    return dist(cur["center"], prev["center"])


def compute_direction(track_frame_map, frame_id):
    cur = track_frame_map.get(frame_id)
    prev = prev_point(track_frame_map, frame_id)
    if cur is None or prev is None:
        return [0.0, 0.0]
    return vector(prev["center"], cur["center"])


def summarize_tracks(tracks):
    out = {}
    for tid, pts in tracks.items():
        out[tid] = {
            "track_length": len(pts),
            "first_frame": pts[0]["frame_id"],
            "last_frame": pts[-1]["frame_id"],
            "first_time": round(pts[0]["time_sec"], 3),
            "last_time": round(pts[-1]["time_sec"], 3),
            "first_center": [round(x, 2) for x in pts[0]["center"]],
            "last_center": [round(x, 2) for x in pts[-1]["center"]],
        }
    return out


In [14]:
# ==============================
# 사람 bbox에서 손/발 후보 bbox 만들기
# ==============================
# pose/keypoint 모델 없이 person bbox만 있을 때의 근사입니다.
# 정확한 손/발 좌표가 아니라 "손/발이 있을 가능성이 높은 영역"으로 생각하면 됩니다.


def part_boxes_from_person_bbox(bbox):
    x1, y1, x2, y2 = map(float, bbox)
    w, h = bbox_wh(bbox)

    # 손 후보: 몸통 중간 높이의 좌우 바깥쪽
    left_hand = [x1 - 0.10*w, y1 + 0.25*h, x1 + 0.32*w, y1 + 0.72*h]
    right_hand = [x2 - 0.32*w, y1 + 0.25*h, x2 + 0.10*w, y1 + 0.72*h]

    # 발 후보: bbox 하단 좌우
    left_foot = [x1 + 0.05*w, y1 + 0.72*h, x1 + 0.50*w, y2 + 0.05*h]
    right_foot = [x1 + 0.50*w, y1 + 0.72*h, x2 - 0.05*w, y2 + 0.05*h]

    return {
        "left_hand": clamp_box(left_hand),
        "right_hand": clamp_box(right_hand),
        "left_foot": clamp_box(left_foot),
        "right_foot": clamp_box(right_foot),
    }


def min_part_to_body_distance(attacker_bbox, target_bbox):
    parts = part_boxes_from_person_bbox(attacker_bbox)
    dists = {name: box_distance(pb, target_bbox) for name, pb in parts.items()}
    best_name = min(dists, key=dists.get)
    return best_name, dists[best_name], parts[best_name]


In [15]:
# ==============================
# 싸움 탐지
# ==============================
# 규칙:
# 1) 두 사람이 가까움
# 2) A의 손/발 후보 bbox가 B의 몸 bbox와 겹치거나 가까움 또는 B의 손/발이 A에게 가까움
# 3) 해당 접촉 후보가 시간적으로 분리되어 2회 이상 반복


def count_separated_hits(hit_signals, min_sep=FIGHT_MIN_SEPARATION_SEC):
    hits = [x for x in hit_signals if x["signal"]]
    selected = []
    last_t = None
    for h in hits:
        if last_t is None or h["time_sec"] - last_t >= min_sep:
            selected.append(h)
            last_t = h["time_sec"]
    return selected


def detect_fight(tracks, frame_map):
    tfm = {tid: get_point_by_frame(pts) for tid, pts in tracks.items()}
    pair_signals = defaultdict(list)

    for frame_id in sorted(frame_map.keys()):
        persons = frame_map[frame_id]
        if len(persons) < 2:
            continue

        for tid_a, tid_b in combinations(sorted(persons.keys()), 2):
            a, b = persons[tid_a], persons[tid_b]
            body_d = dist(a["center"], b["center"])
            if body_d > FIGHT_BODY_DISTANCE_TH:
                continue

            speed_a = compute_speed(tfm[tid_a], frame_id)
            speed_b = compute_speed(tfm[tid_b], frame_id)

            a_part, a_d, a_part_box = min_part_to_body_distance(a["bbox"], b["bbox"])
            b_part, b_d, b_part_box = min_part_to_body_distance(b["bbox"], a["bbox"])

            # 양쪽 중 하나라도 손/발 후보가 상대 bbox에 붙고, 움직임이 있으면 공격 접촉 후보
            a_hit = a_d <= FIGHT_PART_TOUCH_MARGIN and speed_a >= FIGHT_MOTION_TH
            b_hit = b_d <= FIGHT_PART_TOUCH_MARGIN and speed_b >= FIGHT_MOTION_TH
            signal = a_hit or b_hit

            pair_key = tuple(sorted([tid_a, tid_b]))
            pair_signals[pair_key].append({
                "frame_id": frame_id,
                "time_sec": round(a["time_sec"], 3),
                "distance_center": round(body_d, 2),
                "signal": bool(signal),
                "a_track_id": tid_a,
                "b_track_id": tid_b,
                "a_speed": round(speed_a, 2),
                "b_speed": round(speed_b, 2),
                "a_best_part": a_part,
                "a_part_to_b_distance": round(a_d, 2),
                "a_part_bbox": a_part_box,
                "b_best_part": b_part,
                "b_part_to_a_distance": round(b_d, 2),
                "b_part_bbox": b_part_box,
                "a_bbox": clamp_box(a["bbox"]),
                "b_bbox": clamp_box(b["bbox"]),
            })

    events = []
    for pair_key, signals in pair_signals.items():
        if len(signals) < FIGHT_MIN_HIT_COUNT:
            continue

        start_idx = 0
        while start_idx < len(signals):
            start_t = signals[start_idx]["time_sec"]
            window = []
            for j in range(start_idx, len(signals)):
                if signals[j]["time_sec"] - start_t <= FIGHT_WINDOW_SEC:
                    window.append(signals[j])
                else:
                    break

            separated_hits = count_separated_hits(window)
            if len(separated_hits) >= FIGHT_MIN_HIT_COUNT:
                ratio = sum(1 for x in window if x["signal"]) / max(1, len(window))
                events.append({
                    "event_class": "fight",
                    "detected": True,
                    "track_pair": list(pair_key),
                    "start_frame": window[0]["frame_id"],
                    "end_frame": window[-1]["frame_id"],
                    "start_time": window[0]["time_sec"],
                    "end_time": window[-1]["time_sec"],
                    "confidence": round(min(0.95, 0.55 + 0.12 * len(separated_hits) + 0.20 * ratio), 3),
                    "reason": "손/발 후보 bbox가 상대 person bbox와 겹치거나 가까워지는 접촉 후보가 2회 이상 반복됨.",
                    "evidence": {
                        "separated_hit_count": len(separated_hits),
                        "signal_ratio": round(ratio, 3),
                        "sample_hits": separated_hits[:5],
                    }
                })
                start_idx += max(1, len(window))
            else:
                start_idx += 1

    return merge_events(events, event_class="fight")


In [16]:
# ==============================
# 스토킹 탐지
# ==============================
# directed pair로 봅니다.
# follower가 target의 뒤쪽에 있고, 둘이 같은 방향으로 이동하며, 일정 거리 범위를 유지하면 stalking 후보.


def behind_metrics(follower_center, target_center, target_dir):
    n = norm(target_dir)
    if n == 0:
        return 0.0, 999999.0

    unit = [target_dir[0] / n, target_dir[1] / n]
    rel = vector(follower_center, target_center)  # follower -> target

    # dot이 양수면 follower에서 target으로 가는 방향이 target 진행방향과 비슷함 = follower가 뒤쪽에 있음
    behind_dot = rel[0] * unit[0] + rel[1] * unit[1]

    # target 진행방향에 수직인 거리
    lateral = abs(rel[0] * (-unit[1]) + rel[1] * unit[0])
    return behind_dot, lateral


def detect_stalking(tracks, frame_map):
    tfm = {tid: get_point_by_frame(pts) for tid, pts in tracks.items()}
    track_ids = sorted(tracks.keys())
    directed_signals = defaultdict(list)

    for follower_id, target_id in permutations(track_ids, 2):
        common_frames = sorted(set(tfm[follower_id]) & set(tfm[target_id]))
        if len(common_frames) < STALKING_MIN_SIGNAL_COUNT:
            continue

        for fid in common_frames:
            follower = tfm[follower_id][fid]
            target = tfm[target_id][fid]

            follower_dir = compute_direction(tfm[follower_id], fid)
            target_dir = compute_direction(tfm[target_id], fid)
            sim = cosine_similarity(follower_dir, target_dir)

            d = dist(follower["foot_point"], target["foot_point"])
            behind_dot, lateral = behind_metrics(follower["center"], target["center"], target_dir)

            follower_speed = compute_speed(tfm[follower_id], fid)
            target_speed = compute_speed(tfm[target_id], fid)

            signal = (
                STALKING_MIN_DIST <= d <= STALKING_MAX_DIST
                and sim >= STALKING_DIR_SIM_TH
                and behind_dot >= STALKING_BEHIND_DOT_TH
                and lateral <= STALKING_LATERAL_MAX
                and follower_speed > 1.5
                and target_speed > 1.5
            )

            directed_signals[(follower_id, target_id)].append({
                "frame_id": fid,
                "time_sec": round(follower["time_sec"], 3),
                "signal": bool(signal),
                "follower_track_id": follower_id,
                "target_track_id": target_id,
                "distance": round(d, 2),
                "direction_similarity": round(sim, 3),
                "behind_dot": round(behind_dot, 2),
                "lateral_distance": round(lateral, 2),
                "follower_speed": round(follower_speed, 2),
                "target_speed": round(target_speed, 2),
                "follower_bbox": clamp_box(follower["bbox"]),
                "target_bbox": clamp_box(target["bbox"]),
            })

    events = []
    for pair_key, signals in directed_signals.items():
        start_idx = 0
        while start_idx < len(signals):
            start_t = signals[start_idx]["time_sec"]
            window = []
            for j in range(start_idx, len(signals)):
                if signals[j]["time_sec"] - start_t <= STALKING_WINDOW_SEC:
                    window.append(signals[j])
                else:
                    break

            true_signals = [x for x in window if x["signal"]]
            if len(window) >= STALKING_MIN_SIGNAL_COUNT and true_signals:
                ratio = len(true_signals) / len(window)
                distance_std = std([x["distance"] for x in true_signals])

                if ratio >= STALKING_MIN_RATIO and distance_std <= STALKING_DISTANCE_STD_TH:
                    follower_id, target_id = pair_key
                    events.append({
                        "event_class": "stalking",
                        "detected": True,
                        "stalking_type": "behind_following",
                        "follower_track_id": follower_id,
                        "target_track_id": target_id,
                        "start_frame": window[0]["frame_id"],
                        "end_frame": window[-1]["frame_id"],
                        "start_time": window[0]["time_sec"],
                        "end_time": window[-1]["time_sec"],
                        "confidence": round(min(0.93, 0.50 + 0.35 * ratio), 3),
                        "reason": "한 사람이 다른 사람의 뒤쪽에서 일정 거리 범위를 유지하며 같은 방향으로 이동함.",
                        "evidence": {
                            "signal_ratio": round(ratio, 3),
                            "signal_count": len(true_signals),
                            "window_count": len(window),
                            "distance_std": round(distance_std, 3),
                            "sample_frames": true_signals[:5],
                        }
                    })
                    start_idx += max(1, len(window))
                else:
                    start_idx += 1
            else:
                start_idx += 1

    return merge_events(events, event_class="stalking")


In [17]:
# ==============================
# 이벤트 병합 / 최종 판단
# ==============================

def same_event_identity(a, b):
    if a["event_class"] != b["event_class"]:
        return False

    if a["event_class"] == "fight":
        return sorted(a.get("track_pair", [])) == sorted(b.get("track_pair", []))

    if a["event_class"] == "stalking":
        return (
            a.get("follower_track_id") == b.get("follower_track_id")
            and a.get("target_track_id") == b.get("target_track_id")
        )

    return False


def merge_events(events, event_class, frame_gap=30):
    if not events:
        return []
    events = sorted(events, key=lambda x: (x["start_frame"], x["end_frame"], -x.get("confidence", 0)))
    merged = []

    for ev in events:
        if not merged:
            merged.append(ev)
            continue
        last = merged[-1]
        if same_event_identity(last, ev) and ev["start_frame"] <= last["end_frame"] + frame_gap:
            last["end_frame"] = max(last["end_frame"], ev["end_frame"])
            last["end_time"] = max(last["end_time"], ev["end_time"])
            last["confidence"] = max(last["confidence"], ev["confidence"])
            # evidence는 대표 샘플만 유지
        else:
            merged.append(ev)
    return merged


def decide_final(fight_events, stalking_events):
    predictions = sorted(fight_events + stalking_events, key=lambda x: x.get("confidence", 0), reverse=True)
    if not predictions:
        return {
            "detected": False,
            "main_event": "none",
            "confidence": 0.0,
            "predictions": [],
            "reason": "싸움 또는 스토킹 조건을 만족하는 사람 track pair를 찾지 못함."
        }
    main = predictions[0]
    return {
        "detected": True,
        "main_event": main["event_class"],
        "confidence": main["confidence"],
        "predictions": predictions,
    }


In [18]:
# ==============================
# 실행
# ==============================
raw = load_json(INPUT_JSON)
video_name, tracks = build_person_tracks(raw)
frame_map = build_frame_map(tracks)

print("================================")
print("[BD: Fight / Stalking Detection]")
print("================================")
print("video_name:", video_name)
print("person track count:", len(tracks))

if len(tracks) < 2:
    fight_events = []
    stalking_events = []
    final_prediction = {
        "detected": False,
        "main_event": "none",
        "confidence": 0.0,
        "predictions": [],
        "reason": "사람 track이 2개 미만이라 관계 기반 행동 분석 불가. tracking_id가 있는 person 결과가 필요함."
    }
else:
    fight_events = detect_fight(tracks, frame_map)
    stalking_events = detect_stalking(tracks, frame_map)
    final_prediction = decide_final(fight_events, stalking_events)

output = {
    "video_name": video_name,
    "input_file": str(INPUT_JSON),
    "task": "bd_fight_and_stalking_detection",
    "decision_rule": {
        "fight": [
            "person bbox에서 손/발 후보 bbox를 근사",
            "손/발 후보 bbox가 상대 person bbox와 겹치거나 가까운지 확인",
            "해당 접촉 후보가 3초 안에 시간적으로 분리되어 2회 이상 반복되면 fight 후보"
        ],
        "stalking": [
            "directed pair, 즉 follower-target 관계로 검사",
            "follower가 target 진행방향 기준 뒤쪽에 있는지 확인",
            "둘의 이동방향이 유사하고 일정 거리 범위를 유지하는지 확인",
            "5초 구간에서 조건 만족 비율이 기준 이상이면 stalking 후보"
        ]
    },
    "parameters": {
        "min_track_length": MIN_TRACK_LENGTH,
        "fight_body_distance_th": FIGHT_BODY_DISTANCE_TH,
        "fight_part_touch_margin": FIGHT_PART_TOUCH_MARGIN,
        "fight_motion_th": FIGHT_MOTION_TH,
        "fight_window_sec": FIGHT_WINDOW_SEC,
        "fight_min_hit_count": FIGHT_MIN_HIT_COUNT,
        "stalking_min_dist": STALKING_MIN_DIST,
        "stalking_max_dist": STALKING_MAX_DIST,
        "stalking_dir_sim_th": STALKING_DIR_SIM_TH,
        "stalking_window_sec": STALKING_WINDOW_SEC,
        "stalking_min_ratio": STALKING_MIN_RATIO,
    },
    "track_summaries": summarize_tracks(tracks),
    "fight_events": fight_events,
    "stalking_events": stalking_events,
    "final_prediction": final_prediction,
}

save_json(output, OUTPUT_JSON)

print("\n[final_prediction]")
print(json.dumps(final_prediction, ensure_ascii=False, indent=2))
print("\n저장 완료:", OUTPUT_JSON)


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\우성한\\cf2'

In [ ]:
# ==============================
# 결과 빠르게 확인
# ==============================
print("fight_events:", len(fight_events))
print("stalking_events:", len(stalking_events))

for ev in output["final_prediction"].get("predictions", [])[:5]:
    print("-", ev["event_class"], ev.get("confidence"), ev.get("start_frame"), "~", ev.get("end_frame"), ev.get("reason"))
